# Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data

In [ ]:
maison_df = pd.read_csv('datasets_prepd/dvf_maison.csv')
print(maison_df.head())

# Features

In [ ]:
features = [
    "longitude",
    "latitude",
    "code_postal",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "prix_m2_ref",
    "surface_terrain",
    "number_of_lots"
]

target = "valeur_fonciere_actualisee"

X = maison_df[features]
y = maison_df[target]

print("Number of features:", X.shape[1])
print(X.head())
print(y.head())

# Make sets : trail and test and validate

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# Ssave

In [ ]:
os.makedirs("data_maison", exist_ok=True)

train_df = X_train.copy()
val_df = X_val.copy()
test_df = X_test.copy()

train_df[target] = y_train
val_df[target] = y_val
test_df[target] = y_test

train_df.to_csv("data_maison/maison_train.csv", index=False)
val_df.to_csv("data_maison/maison_val.csv", index=False)
test_df.to_csv("data_maison/maison_test.csv", index=False)

print("Datasets saved.")

Train model

In [ ]:
model_maison = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model_maison.fit(X_train, y_train)

print("Model trained.")

Evaluation

In [ ]:
y_val_pred = model_maison.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae = mean_absolute_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print("Validation RMSE:", rmse)
print("Validation MAE:", mae)
print("Validation R2:", r2)

In [8]:
df_eval = pd.DataFrame({
    "prix_reel": y_val,
    "prix_predit": y_val_pred
})

df_eval["erreur_pct"] = abs(df_eval["prix_reel"] - df_eval["prix_predit"]) / df_eval["prix_reel"] * 100

print(df_eval.head())
print("Erreur moyenne (%) :", df_eval["erreur_pct"].mean())

       prix_reel    prix_predit  erreur_pct
45168   459375.0  669942.399225   45.837801
17026   341600.0  260015.692204   23.882994
30940   492200.0  543327.720756   10.387591
33373   135300.0  243846.902182   80.226831
35217   116611.0  169272.044594   45.159586
Erreur moyenne (%) : 20.74344133312875


# Test

In [ ]:
y_train_pred = model_maison.predict(X_train)

rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae = mean_absolute_error(y_train, y_train_pred)
r2 = r2_score(y_train, y_train_pred)

print("Train RMSE:", rmse)
print("Train MAE:", mae)
print("Train R2:", r2)

Save model

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(model_maison, "models/maison_random_forest_model.pkl")

print("Model saved.")

In [ ]:
print("Number of features expected:", model_maison.n_features_in_)
print("Feature names:", features)